In [102]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber

In [103]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [104]:
with open("quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [105]:
def extract_statement(exercise):
    m = re.search("R E S O L U C I Ó N", exercise)
    if m:
        end_pos = m.start()
        exercise = exercise[:end_pos].strip()
        lines = exercise.split("\n")
        exam = lines[-1]
        statement = "\n".join(lines[:-1])
        return exam, statement
    return None

In [106]:
def extract_exercises(file):
    exercises = []
    with pdfplumber.open(file) as pdf:
        for idx, page in enumerate(pdf.pages):
            texto = ext.extract_text_from_page(page)
            if idx > 0:  # ejercicios comienzan en 2da página
                st = extract_statement(texto)
                if st:
                    exercises.append(extract_statement(texto))
    return exercises

In [107]:
def create_df_exercises(folder: Path) -> pd.DataFrame:
    exercises_list = []
    if not isinstance(folder, Path):
        raise ValueError("La función espera un objeto Path con el directorio a procesar.")
        if not folder.is_dir():
            raise ValueError("El objeto Path debe ser un directorio.")
    
    for file in folder.iterdir():   
        exams, statements = [], []
        #print("Processing file: ", file.name)
        exercises = extract_exercises(file)
        for ex in exercises:
            if ex:
                exercises_list.append({"exam" : ex[0], "statement": ex[1]})
    return pd.DataFrame(exercises_list)

In [108]:
def classify_exercise(statement: str, topic: str, keywords = quimica_keywords) -> tuple:
    topic_keywords = keywords[topic]["keywords_tema"]

    exercise_topic = None
    for topic_kw in topic_keywords:
        if topic_kw in statement.lower():
            exercise_topic = topic    
            break

    if exercise_topic:
        exercise_types = []
        exercise_type_keywords = keywords[topic]["tipos_ejercicio"]
        
        for tipo, tipo_keywords in exercise_type_keywords.items():
            if any(kw in statement.lower() for kw in tipo_keywords):
                exercise_types.append(tipo)
    else:
        exercise_topic, exercise_types = None, [] 

    return exercise_topic, exercise_types

Clasificar ejercicios de una carpeta

In [109]:
# # RENOMBRAR CARPETAS PARA QUE COINCIDAN CON CLAVES DEL JSON CON KEYWORDS
# import shutil

# acentos = {"á" : 'a', "é" : 'e', 'í' : 'i', 'ó' : 'o'}
# translation_table = str.maketrans(acentos)

# for folder in Path("temas").iterdir():
#     new_folder = folder.name.lower().replace(" ", "_")
#     new_folder = new_folder.translate(translation_table)
    
#     new_folder = Path("/".join(folder.parts[:-1]) + f"/{new_folder}")
#     shutil.move(folder, new_folder)

Ordenar directorios sin acentos:

In [110]:
TEMAS = sorted(Path("temas").iterdir())

In [111]:
[i.name for i in TEMAS]

['acido_base',
 'cantidad_quimica',
 'configuracion_electronica',
 'enlace_quimico',
 'equilibrio_quimico',
 'formulacion',
 'reactividad_organica',
 'redox',
 'solubilidad',
 'termoquimica']

## Clasificar carpeta de ejercicios

In [112]:
df = create_df_exercises(TEMAS[2])
df["topic"], df["exercise_type"] = zip(*df.statement.apply(classify_exercise, topic=TEMAS[2].name))
df = df.explode("exercise_type")
df = df.sort_values("exam").reset_index(drop=True)
df = df.drop_duplicates()

In [113]:
df.isna().sum()

exam              0
statement         0
topic            16
exercise_type    48
dtype: int64

In [114]:
df

,exam,statement,topic,exercise_type
0,QUIMICA. 2005. RESERVA 1. EJERCICIO 3. OPCIÓN B,"Dadas las siguientes especies: Ar, Ca2+y Cl −....",configuracion_electronica,NaN
1,QUIMICA. 2005. RESERVA 2 EJERCICIO 2 OPCIÓN A,a) Razone si para un electrón son posibles las...,configuracion_electronica,Combinaciones de números cuánticos
2,QUIMICA. 2005. RESERVA 3 EJERCICIO 2 O PCIÓN A,a) Indique el número de electrones desapareado...,configuracion_electronica,electrones desapareados
3,QUIMICA. 2005. RESERVA 3 EJERCICIO 2 O PCIÓN A,a) Indique el número de electrones desapareado...,configuracion_electronica,Combinaciones de números cuánticos
4,QUIMICA. 2005. RESERVA 4 EJERCICIO 2 OPCIÓN A,"Indique: a) Los subniveles de energía, dados p...",configuracion_electronica,NaN
...,...,...,...,...
306,QUÍMICA. 2026. RESERVA 3. EJERCICIO 1B,Razone si las siguientes afirmaciones son verd...,configuracion_electronica,electrones desapareados
307,QUÍMICA. 2026. RESERVA 3. EJERCICIO 1B,Razone si las siguientes afirmaciones son verd...,configuracion_electronica,verdadero o falso
308,QUÍMICA. 2026. RESERVA 4. EJERCICIO 1A,El ion X 2 − tiene Z = 1 6 y A = 3 2 :\na) Esc...,configuracion_electronica,Combinaciones de números cuánticos
309,QUÍMICA. 2026. RESERVA 4. EJERCICIO 1A,El ion X 2 − tiene Z = 1 6 y A = 3 2 :\na) Esc...,configuracion_electronica,Composición atómica


In [115]:
df.exercise_type.unique()

<StringArray>
[                                                    nan,
                    'Combinaciones de números cuánticos',
                               'electrones desapareados',
                                   'Composición atómica',
                                              'isótopos',
 'Situar elementos en tabla periódica (grupo y período)',
                                       'Ion más estable',
                                     'verdadero o falso',
                      'Capacidad de niveles y orbitales',
                                       'Isoelectrónicos',
                         'Comparar energía de orbitales',
                                  'Identificar elemento']
Length: 12, dtype: str

In [116]:
df[df.exercise_type.isna()]

,exam,statement,topic,exercise_type
0,QUIMICA. 2005. RESERVA 1. EJERCICIO 3. OPCIÓN B,"Dadas las siguientes especies: Ar, Ca2+y Cl −....",configuracion_electronica,NaN
4,QUIMICA. 2005. RESERVA 4 EJERCICIO 2 OPCIÓN A,"Indique: a) Los subniveles de energía, dados p...",configuracion_electronica,NaN
5,QUIMICA. 2012. RESERVA 1. EJERCICIO 2. OPCIÓN B,Escriba la configuración electrónica correspon...,configuracion_electronica,NaN
6,QUIMICA. 2012. RESERVA 2. EJERCICIO 2. OPCIÓN A,Indique razonadamente:\na) Cómo evoluciona la ...,configuracion_electronica,NaN
13,QUIMICA. 2017. SEPTIEMBRE. EJERCICIO 2 OPCIÓN A,"a) Justifique cuál de las siguientes especies,...",NaN,NaN
25,QUÍMICA. 2000. RESERVA 1. EJERCICIO 4. OPCIÓN A,"Los elementos Na, Al, y Cl tienen números atóm...",configuracion_electronica,NaN
27,QUÍMICA. 2000. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Los elementos A y B tienen, en sus últimos niv...",NaN,NaN
29,QUÍMICA. 2000. SEPTIEMBRE. EJERCICI O 2. OPCIÓN B,Las dos tablas siguientes corresponden a radio...,configuracion_electronica,NaN
36,QUÍMICA. 2001. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,Defina: a) Energía de ionización. b) Afinidad ...,NaN,NaN
37,QUÍMICA. 2002. JUNIO. EJERCICIO 2. OPCIÓN A,a) Defina afinidad electrónica.\nb) ¿Qué crite...,configuracion_electronica,NaN


In [117]:
df_propiedades = df[df.topic.isna()]

In [118]:
df_propiedades["topic"], df_propiedades["exercise_type"] = zip(*df_propiedades.statement.apply(classify_exercise, topic="propiedades_periodicas"))
df_propiedades = df_propiedades.explode("exercise_type")
df_propiedades = df_propiedades.sort_values("exam").reset_index(drop=True)
df_propiedades = df_propiedades.drop_duplicates()

In [119]:
df_propiedades

,exam,statement,topic,exercise_type
0,QUIMICA. 2017. SEPTIEMBRE. EJERCICIO 2 OPCIÓN A,"a) Justifique cuál de las siguientes especies,...",propiedades_periodicas,Propiedades periódicas
1,QUÍMICA. 2000. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Los elementos A y B tienen, en sus últimos niv...",propiedades_periodicas,Propiedades periódicas
2,QUÍMICA. 2001. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,Defina: a) Energía de ionización. b) Afinidad ...,propiedades_periodicas,Propiedades periódicas
3,QUÍMICA. 2004. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,"Considere la serie de elementos: Li, Na, K, Rb...",propiedades_periodicas,Propiedades periódicas
4,QUÍMICA. 2009. JUNIO. EJERCICIO 2. OPCIÓN A,La siguiente tabla proporciona los valores de ...,propiedades_periodicas,Propiedades periódicas
5,QUÍMICA. 2009. JUNIO. EJERCICIO 2. OPCIÓN A,La siguiente tabla proporciona los valores de ...,propiedades_periodicas,Comparación energías de ionización
6,QUÍMICA. 2013. JUNIO. EJERCICIO 2. OPCIÓN A,"Para los siguientes elementos Na, P, S y Cl, d...",propiedades_periodicas,Propiedades periódicas
7,QUÍMICA. 2013. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Dado los elementos Cl, K y Ar, ordene razonada...",propiedades_periodicas,Propiedades periódicas
8,QUÍMICA. 2015. RESERVA 2. EJERCICIO 2. OPCIÓN A,Razone si las siguientes afirmaciones son verd...,propiedades_periodicas,verdadero o falso
9,QUÍMICA. 2015. RESERVA 2. EJERCICIO 2. OPCIÓN A,Razone si las siguientes afirmaciones son verd...,propiedades_periodicas,Propiedades periódicas


In [120]:
df_propiedades.isna().sum()

exam             0
statement        0
topic            0
exercise_type    0
dtype: int64

In [121]:
df = pd.merge(df, df_propiedades, on=["exam", "statement"])

In [131]:
df.isna().sum()

exam             0
statement        0
topic            0
exercise_type    0
dtype: int64

In [123]:
df["topic"] = df.apply(lambda row: row["topic_y"] if pd.isna(row["topic_x"]) else row["topic_x"], axis=1)
df["exercise_type"] = df.apply(lambda row: row["exercise_type_y"] if pd.isna(row["exercise_type_x"]) else row["exercise_type_x"], axis=1)

In [124]:
df = df.drop([i for i in df.columns if ("_x" in i) or ("_y" in i)], axis = 1)

In [130]:
df.sort_values("exam")

,exam,statement,topic,exercise_type
0,QUIMICA. 2017. SEPTIEMBRE. EJERCICIO 2 OPCIÓN A,"a) Justifique cuál de las siguientes especies,...",propiedades_periodicas,Propiedades periódicas
1,QUÍMICA. 2000. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Los elementos A y B tienen, en sus últimos niv...",propiedades_periodicas,Propiedades periódicas
2,QUÍMICA. 2001. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,Defina: a) Energía de ionización. b) Afinidad ...,propiedades_periodicas,Propiedades periódicas
3,QUÍMICA. 2004. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,"Considere la serie de elementos: Li, Na, K, Rb...",propiedades_periodicas,Propiedades periódicas
4,QUÍMICA. 2009. JUNIO. EJERCICIO 2. OPCIÓN A,La siguiente tabla proporciona los valores de ...,propiedades_periodicas,Propiedades periódicas
5,QUÍMICA. 2009. JUNIO. EJERCICIO 2. OPCIÓN A,La siguiente tabla proporciona los valores de ...,propiedades_periodicas,Comparación energías de ionización
6,QUÍMICA. 2013. JUNIO. EJERCICIO 2. OPCIÓN A,"Para los siguientes elementos Na, P, S y Cl, d...",propiedades_periodicas,Propiedades periódicas
7,QUÍMICA. 2013. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Dado los elementos Cl, K y Ar, ordene razonada...",propiedades_periodicas,Propiedades periódicas
8,QUÍMICA. 2015. RESERVA 2. EJERCICIO 2. OPCIÓN A,Razone si las siguientes afirmaciones son verd...,propiedades_periodicas,verdadero o falso
9,QUÍMICA. 2015. RESERVA 2. EJERCICIO 2. OPCIÓN A,Razone si las siguientes afirmaciones son verd...,propiedades_periodicas,Propiedades periódicas


In [128]:
df.exam = df.exam.replace("QUIMICA", "QUÍMICA")